# M3 step 11: RQ3, does stamping change the detectors? (tasks 7 and 9)

RQ3 asks whether adding the stamp changes the detectors' scores or verdicts. The proposal expects the
frequency based M1 to react more than the CLIP based M2, because the stamp is a small high-frequency
pattern.

For each Real and AI photo from the GenImage test split:

1. the photo is saved as PNG and scored by every detector (**before**),
2. it is stamped with `embed`, using the verdict and P(AI) the pipeline would stamp, and saved as PNG,
3. the stamped PNG is scored again (**after**).

Both versions go through exactly the same steps before each detector: same PNG format, and each
detector's own preprocessing (for M1: its JPEG step with the same per-image quality, the same 256 tile
and the same centre 224 crop). The image id is the same for both, so nothing differs except the stamp.
`embed` keeps the photo's size, so no extra resize-only control arm is needed.

Reported per detector: mean change in P(AI) for Real and for AI photos (with a 95% bootstrap interval),
the number of verdicts that flipped and how, and accuracy and AUROC before and after.

**Now (task 7):** M2 and fusion are not ready, so the dummy detector stands in and the notebook is
tested end to end. **When the models arrive (task 9):** switch them on below; nothing else changes.

**GPU:** helpful, not required. ResNet-50 and CLIP on 600 photos x 2 take a few minutes on a GPU and
about 20 to 40 minutes on a CPU.

## Setup

**Kaggle:** turn on *Settings > Accelerator > GPU T4 x2* (only notebooks 8, 9 and 11 need it) and
*Internet on*, and attach the `veristamp-m3-inputs` dataset (README.md, "Running on Kaggle"). The first
cell then finds the code, installs what is missing and moves into `m3_experiments`. No path to edit.

**Colab or a local machine:** set the paths once in `settings.py`, then run from the `m3_experiments`
folder.

In [ ]:
# Kaggle setup: find the dataset wherever Kaggle put it, copy the code, point the notebooks at the files.
import os, shutil, subprocess, sys
from pathlib import Path

def find(rel, max_depth=6):
    for depth in range(max_depth + 1):
        hits = sorted(Path('/kaggle/input').glob('/'.join(['*'] * depth + [rel])))
        if hits:
            return hits[0]
    return None

if Path('/kaggle/input').is_dir() and not Path('settings.py').exists():
    code = find('m3_experiments/settings.py')
    model = find('m3/best.pt')
    print('Code found at: ', code)
    print('Model found at:', model)
    if code is None or model is None:
        print(subprocess.run(['find', '/kaggle/input', '-maxdepth', '4'], capture_output=True, text=True).stdout[:3000])
        raise SystemExit('The veristamp-m3-inputs dataset is not attached, or has a different layout (see the list above).')
    repo = Path('/kaggle/working/VeriStamp')
    if not (repo / 'm3_experiments' / 'settings.py').exists():
        shutil.rmtree(repo, ignore_errors=True)          # removes a failed earlier clone
        shutil.copytree(code.parent.parent, repo, ignore=shutil.ignore_patterns('.git'))
    # Each item is found on its own, so the inputs can come in several zips or datasets.
    paths = {'M3E_MODEL_PATH': 'm3/best.pt', 'M3E_TEST_DIR': 'm3/test', 'M3E_TEST_MANIFEST': 'm3/test_manifest.csv',
             'M3E_GENIMAGE_SPLIT_DIR': 'genimage/splits', 'M3E_GENIMAGE_ROOT': 'genimage/images',
             'M3E_M1_CHECKPOINT': 'm1/best.pt', 'M3E_M1_VSLIB_DIR': 'm1/vslib_m1_final'}
    for name, rel in paths.items():
        hit = find(rel)
        os.environ[name] = str(hit if hit else model.parent.parent / rel)   # the notebooks run from here inherit these
        print(f'  {rel:<22} {"found" if hit else "not in the inputs"}')
    os.environ['M3E_WORK_DIR'] = '/kaggle/working/m3_work'
    os.chdir(repo / 'm3_experiments')
    for pkg in ('diffusers', 'lpips'):
        try:
            __import__(pkg)
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
print('Working in', Path.cwd())

In [ ]:
# On Colab instead, uncomment and run once per session:
# from google.colab import drive; drive.mount('/content/drive')
# !git clone https://github.com/KruthikaVirupakshappa/VeriStamp.git /content/VeriStamp
# %cd /content/VeriStamp/m3_experiments
# !pip install -q diffusers transformers accelerate scikit-image lpips

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt

import settings as S
import m3_eval as E
mw = S.setup()                      # imports ../m3_watermark/m3_watermark.py as it is
torch.manual_seed(S.SEED)
np.random.seed(S.SEED)

m3 = mw.load_m3(S.MODEL_PATH)       # load once, reuse for every photo
DEVICE = m3.device
print('Model epoch', m3.epoch, 'on', DEVICE)

In [ ]:
torch.backends.cudnn.deterministic = True     # the same photo must give the same score twice
torch.backends.cudnn.benchmark = False

## Detectors

In [ ]:
from detectors import DummyDetector, M1Detector, M2Detector, DetectorSet, late_fusion

# Switch each detector on when it is ready. With everything False, the dummy detector stands in for
# all of them, so the notebook runs end to end now (task 7) and only these switches change later.
# kaggle_run_all.ipynb sets these through environment variables; otherwise write True or False here.
import os
USE_M1 = os.environ.get('M3E_USE_M1', '0') == '1'           # M1 is trained (README_M1.md)
USE_M2 = os.environ.get('M3E_USE_M2', '0') == '1'           # waiting for Supriya
USE_FUSION = os.environ.get('M3E_USE_FUSION', '0') == '1'   # late fusion (average) until a trained fusion model exists

detectors = {}
if USE_M1:
    detectors['m1_resnet'] = M1Detector(S.M1_VSLIB_DIR, S.M1_CHECKPOINT)
if USE_M2:
    detectors['m2_clip'] = M2Detector()
if not detectors:
    detectors['dummy'] = DummyDetector()
detector_set = DetectorSet(detectors, fusion=late_fusion if USE_FUSION else None)
DECIDE_WITH = 'fusion' if 'fusion' in detector_set.names else detector_set.names[0]
print('Detectors:', detector_set.names, '| verdict made by:', DECIDE_WITH)
if 'dummy' in detectors:
    print('DUMMY DETECTOR: these numbers test the code only. Do not report them.')

## Photos

In [ ]:
N_PER_CLASS = 300
sample = E.load_genimage_test(S.GENIMAGE_SPLIT_DIR, S.GENIMAGE_ROOT, N_PER_CLASS, S.SEED, min_side=224)
print(sample.groupby(['label', 'generator']).size())

## Check that scoring is repeatable

The same photo scored twice must give exactly the same P(AI). Otherwise a score change could come from
randomness and not from the stamp.

In [ ]:
for _, s in sample.head(5).iterrows():
    png = E.save_as(E.open_rgb(s['path']), 'PNG')
    a, b = detector_set(png, str(s['image_id'])), detector_set(png, str(s['image_id']))
    assert a == b, f'scores differ on a repeat: {a} vs {b}'
print('Scores are repeatable.')

## Before and after stamping

In [ ]:
from veristamp_pipeline import DETECTOR_IDS, image_tag_for
rows = []
for _, s in E.tqdm(sample.iterrows(), total=len(sample)):
    image_id = str(s['image_id'])
    before_img = E.save_as(E.open_rgb(s['path']), 'PNG')
    before = detector_set(before_img, image_id)
    lo, hi = S.THRESHOLDS.get(DECIDE_WITH, (0.35, 0.65))
    verdict = E.to_verdict(before[DECIDE_WITH], lo, hi)
    stamped = E.save_as(mw.embed(m3, before_img, verdict, before[DECIDE_WITH],
                                 detector_id=DETECTOR_IDS.get(DECIDE_WITH, 0), image_tag=image_tag_for(image_id)), 'PNG')
    after = detector_set(stamped, image_id)
    stamp_ok = mw.extract(m3, stamped)['valid']
    for det in before:
        rows.append({'image_id': image_id, 'label': int(s['label']), 'generator': s['generator'], 'detector': det,
                     'p_before': before[det], 'p_after': after[det], 'stamped_verdict': verdict,
                     'stamp_valid': stamp_ok, 'psnr': E.psnr(stamped, before_img)})
scores = pd.DataFrame(rows)
print('Stamps readable after stamping:', f"{scores.drop_duplicates('image_id').stamp_valid.mean():.1%}")

## RQ3 table

In [ ]:
summary = E.rq3_summary(scores, S.THRESHOLDS)
cols = ['detector', 'score_change_real', 'ci95_real_low', 'ci95_real_high', 'score_change_ai', 'ci95_ai_low',
        'ci95_ai_high', 'mean_abs_change',
        'verdict_flips', 'real_to_ai', 'ai_to_real', 'into_uncertain', 'out_of_uncertain',
        'accuracy_before', 'accuracy_after', 'auroc_before', 'auroc_after']
summary[[c for c in cols if c in summary]]

In [ ]:
fig, axes = plt.subplots(1, len(summary), figsize=(5 * len(summary), 4), squeeze=False)
for ax, det in zip(axes[0], summary['detector']):
    g = scores[scores.detector == det]
    for label, name in [(0, 'Real'), (1, 'AI')]:
        ax.hist((g.p_after - g.p_before)[g.label == label], bins=40, alpha=0.6, label=name)
    ax.axvline(0, color='k', linewidth=0.8)
    ax.set_title(det); ax.set_xlabel('P(AI) after - before'); ax.legend()
plt.tight_layout()
(S.RESULTS_DIR / 'figures').mkdir(exist_ok=True)
plt.savefig(S.RESULTS_DIR / 'figures' / 'rq3_score_change.png', dpi=120, bbox_inches='tight')
plt.show()

## Save

In [ ]:
stem = 'rq3_detectors' + ('_DUMMY' if 'dummy' in detectors else '')
scores.to_csv(S.RESULTS_DIR / f'{stem}_per_photo.csv', index=False)
summary.to_csv(S.RESULTS_DIR / f'{stem}_summary.csv', index=False)
(S.RESULTS_DIR / f'{stem}_summary.md').write_text(E.to_markdown(summary[[c for c in cols if c in summary]]) + '\n')
E.save_results(S.RESULTS_DIR, stem, meta={'detectors': detector_set.names, 'decide_with': DECIDE_WITH,
               'thresholds': S.THRESHOLDS, 'n_per_class': N_PER_CLASS, 'fusion': 'late (average)' if USE_FUSION else None,
               'environment': E.environment_info()})
print('Saved', stem)